# Gold Layer — KPI Analysis
Pre-aggregated hourly KPIs by zone. This is what feeds the Streamlit dashboard
and the Redis cache. SparkSQL makes the queries look exactly like production
Databricks SQL or Athena queries.

In [ ]:
import sys
sys.path.insert(0, '..')

from storage.spark_session import build_delta_spark
from storage.delta_config import GOLD_ZONE_DEMAND_TABLE

spark = build_delta_spark('GoldKPIAnalysis')

In [ ]:
gold = spark.read.format('delta').load(GOLD_ZONE_DEMAND_TABLE)
gold.printSchema()
print(f'Gold zone_demand rows: {gold.count():,}')

## SparkSQL — Top Revenue Zones

In [ ]:
gold.createOrReplaceTempView('zone_demand')

spark.sql("""
    SELECT
        city_zone,
        SUM(ride_count)           AS total_rides,
        SUM(completed_rides)      AS completed_rides,
        SUM(cancelled_rides)      AS cancelled_rides,
        ROUND(SUM(gross_revenue_inr), 2)  AS total_revenue_inr,
        ROUND(AVG(avg_surge_multiplier), 3) AS avg_surge
    FROM zone_demand
    GROUP BY city_zone
    ORDER BY total_revenue_inr DESC
""").show()

## Peak vs Off-Peak Revenue

In [ ]:
spark.sql("""
    SELECT
        CASE
            WHEN event_hour BETWEEN 7 AND 10 THEN 'morning_peak'
            WHEN event_hour BETWEEN 17 AND 21 THEN 'evening_peak'
            WHEN event_hour BETWEEN 0  AND  5 THEN 'night'
            ELSE 'off_peak'
        END AS period,
        SUM(ride_count)                      AS rides,
        ROUND(SUM(gross_revenue_inr), 2)     AS revenue_inr,
        ROUND(AVG(avg_surge_multiplier), 3)  AS avg_surge
    FROM zone_demand
    GROUP BY 1
    ORDER BY revenue_inr DESC
""").show()

## Cancellation Rate by Zone

In [ ]:
cancellation = spark.sql("""
    SELECT
        city_zone,
        SUM(ride_count)       AS total,
        SUM(cancelled_rides)  AS cancelled,
        ROUND(100.0 * SUM(cancelled_rides) / NULLIF(SUM(ride_count), 0), 2) AS cancel_rate_pct
    FROM zone_demand
    GROUP BY city_zone
    ORDER BY cancel_rate_pct DESC
""").toPandas()

import matplotlib.pyplot as plt

cancellation.plot(
    kind='barh', x='city_zone', y='cancel_rate_pct',
    title='Cancellation Rate by Zone (%)', figsize=(9, 5), color='tomato'
)
plt.tight_layout()
plt.show()
cancellation

## Hourly Revenue Heatmap (zone × hour)

In [ ]:
import pandas as pd
import seaborn as sns

pivot_df = spark.sql("""
    SELECT city_zone, event_hour, ROUND(SUM(gross_revenue_inr), 0) AS revenue
    FROM zone_demand
    GROUP BY city_zone, event_hour
""").toPandas()

pivot = pivot_df.pivot(index='city_zone', columns='event_hour', values='revenue').fillna(0)

plt.figure(figsize=(16, 5))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.5, annot=True, fmt='.0f')
plt.title('Revenue Heatmap — Zone × Hour of Day (INR)')
plt.tight_layout()
plt.show()